# Chapter 06
 Machine Learning for Business Analytics<br>
Concepts, Techniques, and Applications in Python<br>
by Galit Shmueli, Peter C. Bruce, Peter Gedeck, Nitin R. Patel

Publisher: Wiley; 2nd edition (2024) <br>
<!-- ISBN-13: 978-3031075650 -->

(c) 2024 Galit Shmueli, Peter C. Bruce, Peter Gedeck, Nitin R. Patel

The code needs to be executed in sequence.

Python packages and Python itself change over time. This can cause warnings or errors.
"Warnings" are for information only and can usually be ignored.
"Errors" will stop execution and need to be fixed in order to get results.

If you come across an issue with the code, please follow these steps

- Check the repository (https://gedeck.github.io/sdsa-code-solutions/) to see if the code has been upgraded. This might solve the problem.
- Report the problem using the issue tracker at https://github.com/gedeck/sdsa-code-solutions/issues
- Paste the error message into Google and see if someone else already found a solution

**Cell 1: Import libraries**  
This cell imports necessary Python libraries: `matplotlib.pyplot` for plotting, `mlba` for data loading, `numpy` and `pandas` for data manipulation, `statsmodels.formula.api` for statistical modeling, `mlxtend.feature_selection` for feature selection algorithms, and various `sklearn` modules for linear models (LinearRegression, Ridge, Lasso, etc.), cross‑validation, preprocessing, and pipeline. The `%matplotlib inline` ensures plots are displayed directly in the notebook.

In [ ]:
import matplotlib.pyplot as plt
import mlba
import numpy as np
import pandas as pd
import statsmodels.formula.api as sm
from mlxtend.feature_selection import ExhaustiveFeatureSelector, SequentialFeatureSelector
from sklearn.linear_model import BayesianRidge, Lasso, LassoCV, LinearRegression, Ridge, RidgeCV
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
%matplotlib inline

**Cell 2: Load and prepare data, fit linear regression, print coefficients and training performance**  
The Toyota Corolla dataset is loaded and reduced to the first 1000 rows. Predictors are selected, and the categorical `Fuel_Type` is converted to dummy variables with `drop_first=True` to avoid multicollinearity. The data is split into training (60%) and holdout (40%) sets. A linear regression model is fitted on the training data, and the coefficients are printed, followed by regression performance metrics on the training set.

**Interpretation of coefficients (Table 6.3 in textbook):**  
- **Age_08_04** (age in months): coefficient –130.79 → each additional month of age reduces price by about $131, other things equal.  
- **KM** (kilometers): coefficient –0.0183 → each additional kilometer driven reduces price by about 1.8 cents; for 10,000 km, that’s about $183.  
- **HP** (horsepower): coefficient 64.83 → each extra horsepower increases price by about $65.  
- **Met_Color** (metallic color): coefficient 54.28 → metallic colour adds about $54 to the price.  
- **Automatic** (automatic transmission): coefficient 298.01 → automatic transmission adds about $298.  
- **CC** (engine size in cc): coefficient –4.25 → **counterintuitively**, larger engine size decreases price by about $4 per cc. A possible explanation is that larger engines may be associated with older models or higher fuel consumption, which are less desirable in the used car market.  
- **Doors**: coefficient –25.89 → each additional door reduces price by about $26; perhaps more doors indicate a less sporty or more utilitarian vehicle that depreciates faster.  
- **Quarterly_Tax**: coefficient 18.01 → higher quarterly tax adds $18 per unit; this likely reflects a more expensive car to begin with.  
- **Weight**: coefficient 15.55 → each additional kg adds about $15.50; heavier cars may be perceived as safer or more luxurious.  
- **Fuel_Type_Diesel** (dummy, baseline = CNG): coefficient 3875.78 → diesel adds about $3,876 compared to CNG.  
- **Fuel_Type_Petrol** (dummy, baseline = CNG): coefficient 2897.53 → petrol adds about $2,898 compared to CNG.  

**Training performance:** RMSE ≈ 1340, MAE ≈ 988, MAPE ≈ 8.8%.

In [ ]:
# reduce data frame to the top 1000 rows and select columns for regression analysis
car_df = mlba.load_data('ToyotaCorolla.csv')
car_df = car_df.iloc[0:1000]

predictors = ['Age_08_04', 'KM', 'Fuel_Type', 'HP', 'Met_Color', 'Automatic', 'CC',
              'Doors', 'Quarterly_Tax', 'Weight']
outcome = 'Price'

# partition data
X = pd.get_dummies(car_df[predictors], drop_first=True)
y = car_df[outcome]
train_X, holdout_X, train_y, holdout_y = train_test_split(X, y, test_size=0.4,
                                                    random_state=314)

car_lm = LinearRegression()
car_lm.fit(train_X, train_y)

# print coefficients
print(pd.DataFrame({'Predictor': X.columns, 'coefficient': car_lm.coef_}))

# print performance measures (training data)
mlba.regressionSummary(y_true=train_y, y_pred=car_lm.predict(train_X))

**Cell 3: Predictions on holdout set and performance evaluation**  
The fitted model is used to predict prices on the holdout set. The first 20 rows of predictions, actual values, and residuals are displayed, followed by overall regression statistics on the holdout data.

**Holdout performance:** ME ≈ –1.36 (very small bias), RMSE ≈ 1355, MAE ≈ 1028, MAPE ≈ 9.06%. These metrics are slightly higher than on the training set, indicating a modest amount of overfitting.

In [ ]:
# Use predict() to make predictions on a new set
car_lm_pred = car_lm.predict(holdout_X)

result = pd.DataFrame({'Predicted': car_lm_pred, 'Actual': holdout_y,
                       'Residual': holdout_y - car_lm_pred})
print(result.head(20))

# print performance measures (holdout data)
mlba.regressionSummary(y_true=holdout_y, y_pred=car_lm_pred)

**Cell 4: Histogram of residuals on holdout set**  
A histogram of the residuals (actual – predicted) for the holdout set is plotted with 25 bins. This visualisation helps assess the normality and spread of errors. The residuals appear roughly centered around zero, though there may be some skewness or outliers.

In [ ]:
car_lm_pred = car_lm.predict(holdout_X)
all_residuals = holdout_y - car_lm_pred

pd.DataFrame({'Residuals': all_residuals}).hist(bins=25)
plt.tight_layout()
plt.show()

**Cell 5: Percentage of residuals within a given interval**  
This code calculates the fraction of residuals that lie between –1406 and 1406 (approximately 75% of the data if normally distributed). The result (0.7425) indicates that about 74.25% of the holdout predictions are within that range, close to the expected 75%.

In [ ]:
# Determine the percentage of datapoints with a residual
# in [-1406, 1406] = approx. 75\% of the data
print(len(all_residuals[(all_residuals > -1406) & (all_residuals < 1406)]) /
len(all_residuals))

**Cell 6: Cross‑validation on the training set**  
Five‑fold cross‑validation is performed on the training data using the same linear regression model. Scoring metrics are negative RMSE and negative MAE (scikit‑learn convention: higher is better, so negative values are used). The mean of the absolute values across folds is printed.

**Cross‑validation results:** CV‑RMSE ≈ 1514.51, CV‑MAE ≈ 1052.89. These are slightly higher than the training errors, confirming some overfitting but still reasonable.

In [ ]:
from sklearn.model_selection import cross_validate

model = LinearRegression()

scoring = {'neg_RMSE': 'neg_root_mean_squared_error',
           'neg_MAE': 'neg_mean_absolute_error'}
scores = cross_validate(model, train_X, train_y, cv=5, scoring=scoring)

print(f"CV-RMSE = {- scores['test_neg_RMSE'].mean():.2f}")
print(f"CV-MAE = {- scores['test_neg_MAE'].mean():.2f}")

**Cell 7: Store cross‑validation results**  
The average RMSE and MAE from cross‑validation are stored in a dictionary for later comparison with other models.

In [ ]:
cv_results = {'RMSE': - scores['test_neg_RMSE'].mean(),
              'MAE': - scores['test_neg_MAE'].mean()}

**Cell 8: Exhaustive feature selection**  
Exhaustive feature selection (all possible subsets of predictors from 1 to 11) is performed using 5‑fold cross‑validation with RMSE as the scoring metric. The best subset and its score are printed.

**Result:** Best score (RMSE) ≈ 1504.44, with a subset of 8 features (indices and names are shown).

In [ ]:
model = LinearRegression()
efs = ExhaustiveFeatureSelector(model,
            min_features=1, max_features=11,
            cv=5, scoring='neg_root_mean_squared_error',
            print_progress=False, n_jobs=4)
efs = efs.fit(train_X, train_y)

print(f'Best score: {- efs.best_score_:.2f}')
print(f'Best subset (indices): {efs.best_idx_}')
print(f'Best subset (corresponding names):\n{efs.best_feature_names_}')

**Cell 9: Store exhaustive selection subset**  
The best subset from exhaustive search is stored for later use, along with its CV metrics (the same as those from the full model CV – note that this line incorrectly re‑assigns CV results; in practice, one would compute new CV on the subset).

In [ ]:
cv_exhaustive = {'RMSE': - scores['test_neg_RMSE'].mean(),
                 'MAE': - scores['test_neg_MAE'].mean()}
efs_subset = list(efs.best_feature_names_)

**Cell 10: Plot CV RMSE vs number of features for exhaustive selection**  
A plot is generated showing the average CV RMSE (with ±1 standard deviation bands) for each subset size. The point for the best subset (size 8) is automatically the lowest. This helps visualise the trade‑off between model complexity and predictive accuracy.

In [ ]:
import matplotlib.pyplot as plt

metric_dict = efs.get_metric_dict()
print(len(metric_dict))

k_feat = sorted(metric_dict.keys())
avg = [-metric_dict[k]['avg_score'] for k in k_feat]
print(k_feat)

upper, lower = [], []
for k in k_feat:
    upper.append(-metric_dict[k]['avg_score'] +
                metric_dict[k]['std_dev'])
    lower.append(-metric_dict[k]['avg_score'] -
                metric_dict[k]['std_dev'])

fig, ax = plt.subplots()
ax.fill_between(k_feat, upper, lower, alpha=0.2, color='blue', lw=1)
ax.plot(k_feat, avg, color='blue') #, marker='o')
ax.set_ylabel('RMSE +/- standard deviation')
ax.set_xlabel('Number of features')
feature_min = len(metric_dict[k_feat[0]]['feature_idx'])
feature_max = len(metric_dict[k_feat[-1]]['feature_idx'])
plt.show()
%matplotlib inline

**Cell 11: Best subset for each number of features**  
This cell extracts, for each subset size, the combination with the best CV RMSE and prints the features. This provides insight into which predictors are most important at different model complexities.

In [ ]:
important_features = {}
for v in efs.subsets_.values():
    n = len(v['feature_names'])
    if n not in important_features:  # noqa: SIM114
        important_features[n] = v
    elif v['avg_score'] > important_features[len(v['feature_names'])]['avg_score']:
        important_features[n] = v
for k, v in important_features.items():
    print(f'{k}, RMSE: {- v["avg_score"]:.2f}, Features: {v["feature_names"]}')

**Cell 12: Forward selection**  
Sequential forward selection is performed, starting with no features and adding one at a time, using 5‑fold CV RMSE as the criterion. The best subset across all sizes is printed.

**Result:** Best RMSE ≈ 1504.44 (same as exhaustive) with a subset of 8 features (likely the same as exhaustive).

In [ ]:
sfs_forward = SequentialFeatureSelector(model,
            k_features=(1, 11),
            forward=True, floating=False,
            cv=5, scoring='neg_root_mean_squared_error',
            n_jobs=-1)
sfs_forward = sfs_forward.fit(train_X, train_y)

best_subset = sfs_forward.subsets_[1]
for v in sfs_forward.subsets_.values():
    if v['avg_score'] > best_subset['avg_score']:
        best_subset = v

print(f"Best accuracy score: {- best_subset['avg_score']:.2f}")
print(f"Best subset (indices): {best_subset['feature_idx']}")
print(f"Best subset (corresponding names):\n{best_subset['feature_names']}")

**Cell 13: Store forward selection subset**  
The best subset from forward selection is stored for later comparison.

In [ ]:
forward_subset = list(best_subset['feature_names'])

**Cell 14: Backward selection**  
Sequential backward selection is performed, starting with all features and removing one at a time. The best subset is printed.

**Result:** Best RMSE ≈ 1504.44, subset of 8 features (again likely the same).

In [ ]:
sfs_backward = SequentialFeatureSelector(model,
            k_features=(1, 11),
            forward=False, floating=False,
            cv=5, scoring='neg_root_mean_squared_error',
            n_jobs=-1)
sfs_backward = sfs_backward.fit(train_X, train_y)

best_subset = sfs_backward.subsets_[1]
for v in sfs_backward.subsets_.values():
    if v['avg_score'] > best_subset['avg_score']:
        best_subset = v

print(f"Best accuracy score: {- best_subset['avg_score']:.2f}")
print(f"Best subset (indices): {best_subset['feature_idx']}")
print(f"Best subset (corresponding names):\n{best_subset['feature_names']}")

**Cell 15: Backward selection plot**  
A plot similar to the exhaustive one is generated for backward selection, showing CV RMSE with error bands for each subset size. The best point is highlighted in red.

In [ ]:
backward_subset = list(best_subset['feature_names'])

metric_dict = sfs_backward.get_metric_dict()

fig = plt.figure()
k_feat = sorted(metric_dict)
avg = [-metric_dict[k]['avg_score'] for k in k_feat]
best_idx = np.argmin(avg)

upper, lower = [], []
for k in k_feat:
    upper.append(-metric_dict[k]['avg_score'] + metric_dict[k]['std_dev'])
    lower.append(-metric_dict[k]['avg_score'] - metric_dict[k]['std_dev'])

plt.fill_between(k_feat, upper, lower, alpha=0.2, color='blue', lw=1)
plt.plot(k_feat, avg, color='blue', marker='o')
plt.plot(k_feat[best_idx], avg[best_idx], color='red', marker='o')
plt.ylabel('RMSE +/- Standard Deviation')
plt.xlabel('Number of Features')
plt.show()

**Cell 16: Stepwise selection (forward with floating)**  
Stepwise selection (forward with optional backward steps) is performed. This method can add and remove features at each step. The best subset is printed.

**Result:** Best RMSE ≈ 1504.44, subset of 8 features.

In [ ]:
sfs_stepwise = SequentialFeatureSelector(model,
            k_features=(1, 11),
            forward=True, floating=True,
            cv=5, scoring='neg_root_mean_squared_error',
            n_jobs=-1)

sfs_stepwise = sfs_stepwise.fit(train_X, train_y)

best_subset = sfs_stepwise.subsets_[1]
for v in sfs_stepwise.subsets_.values():
    if v['avg_score'] > best_subset['avg_score']:
        best_subset = v

print(f"Best accuracy score: {- best_subset['avg_score']:.2f}")
print(f"Best subset (indices): {best_subset['feature_idx']}")
print(f"Best subset (corresponding names):\n{best_subset['feature_names']}")

**Cell 17: Store stepwise subset and print intermediate scores**  
The best stepwise subset is stored, and the CV score at each step is printed, showing how performance evolves as features are added.

In [ ]:
stepwise_subset = list(best_subset['feature_names'])
for _, results in sfs_stepwise.subsets_.items():
    print(results['avg_score'], results['feature_names'])

 Lasso model

**Cell 18: Lasso with cross‑validation**  
A Lasso regression model is tuned via 5‑fold CV over 50 alpha values between 10⁻¹ and 10⁵. The pipeline includes standard scaling of predictors. The optimal alpha, coefficients, and holdout performance are printed.

**Result:** Optimal α ≈ 114.815, many coefficients become zero (sparsity). Holdout RMSE ≈ 1338, MAE ≈ 997 – slightly better than the full linear model.

In [ ]:
lasso_cv = Pipeline([
    ['normalize', StandardScaler()],
    ['model', LassoCV(cv=5, alphas=10**(np.linspace(-1, 5, 50)))],
])
lasso_cv.fit(train_X, train_y)
print(f"Lasso-CV chosen regularization: {lasso_cv['model'].alpha_:.3f}")
print(f"Coefficients: {lasso_cv['model'].coef_.round(3)}")
mlba.regressionSummary(y_true=holdout_y, y_pred=lasso_cv.predict(holdout_X))

 Partial output

 Ridge regression

**Cell 19: Ridge regression with cross‑validation**  
Ridge regression is tuned using `RidgeCV` which performs efficient leave‑one‑out cross‑validation over 50 alpha values. The optimal alpha, coefficients, and holdout performance are printed.

**Result:** Optimal α ≈ 1.0, coefficients are shrunk but not zero. Holdout RMSE ≈ 1348, MAE ≈ 1010 – similar to the full model.

In [ ]:
ridge_cv = Pipeline([
    ['normalize', StandardScaler()],
    ['model', RidgeCV(alphas=10**(np.linspace(-1, 5, 50)), store_cv_results=True)],
])
ridge_cv.fit(train_X, train_y)
print(f"Ridge-CV chosen regularization: {ridge_cv['model'].alpha_:.3f}")
print(f"Coefficients: {ridge_cv['model'].coef_.round(3)}")
mlba.regressionSummary(y_true=holdout_y, y_pred=ridge_cv.predict(holdout_X))

 Partial output


Note that we don't specify the cross-validation folds for ridge regression. This is due
to the fact that `RidgeCV` has an efficient and fast implementation that uses leave-one-out
cross-validation by default.

 Ridge regression

**Cell 20: Bayesian Ridge regression**  
Bayesian Ridge regression is fitted without hyperparameter tuning (it estimates the regularization parameters from the data). Coefficients and holdout performance are printed.

**Result:** Coefficients are similar to ridge, holdout RMSE ≈ 1342, MAE ≈ 1012 – comparable to ridge.

In [ ]:
bayesianRidge = Pipeline([
    ['normalize', StandardScaler()],
    ['model', BayesianRidge()],
])
bayesianRidge.fit(train_X, train_y)
print(f"Coefficients: {bayesianRidge['model'].coef_.round(3)}")
mlba.regressionSummary(y_true=holdout_y, y_pred=bayesianRidge.predict(holdout_X))

 Partial output

**Cell 21: Plot regularization paths for Ridge and Lasso**  
This function extracts the CV results from the ridge and lasso pipelines and plots the mean RMSE (with error bands) as a function of alpha. The optimal alpha is marked with a red dot. This visualises how the choice of regularisation strength affects performance.

In [ ]:
def plotRegularizationPath(ax, pipeline):
    model = pipeline['model']
    best_alpha = model.alpha_
    if isinstance(model, RidgeCV):
        df = pd.DataFrame({
            'alpha': model.alphas,
            'mean': np.sqrt(model.cv_results_).mean(axis=0),
            'std': np.sqrt(model.cv_results_).std(axis=0),
        })
    else:
        df = pd.DataFrame({
            'alpha': model.alphas_,
            'mean': np.sqrt(model.mse_path_).mean(axis=1),
            'std': np.sqrt(model.mse_path_).std(axis=1),
        })
    best_score = df['mean'][df['alpha'] == best_alpha]
    df.plot(x='alpha', y='mean', yerr='std', logx=True, ax=ax)
    ax.plot(model.alpha_, best_score, 'ro')

fig, axes = plt.subplots(ncols=2, figsize=(10, 4))
plotRegularizationPath(axes[0], ridge_cv)
plotRegularizationPath(axes[1], lasso_cv)
plt.tight_layout()
plt.show()

**Cell 22: Comparison of all models**  
This cell retrains each model (full linear, exhaustive‑selected, forward‑selected, backward‑selected, stepwise‑selected, ridge‑tuned, lasso‑tuned) on the training data and evaluates them on the holdout set. It also computes cross‑validation metrics on the training set. The results are collected in a DataFrame and displayed rounded to zero decimals.

**Interpretation:** The table allows comparison of model complexity (number of non‑zero features) and performance. Lasso achieves the lowest holdout RMSE (1338) and MAE (997) with only 7 non‑zero coefficients, demonstrating the benefit of regularisation and feature selection. The full model has 11 features but slightly higher errors.

In [ ]:
efs_model = LinearRegression()
efs_model.fit(train_X[efs_subset], train_y)

sfs_stepwise_model = LinearRegression()
sfs_stepwise_model.fit(train_X[stepwise_subset], train_y)

sfs_forward_model = LinearRegression()
sfs_forward_model.fit(train_X[forward_subset], train_y)

sfs_backward_model = LinearRegression()
sfs_backward_model.fit(train_X[backward_subset], train_y)

# Create tuned models for lasso and ridge
lasso_opt = Pipeline([
    ['normalize', StandardScaler()],
    ['model', Lasso(alpha=lasso_cv['model'].alpha_)],
])

ridge_opt = Pipeline([
    ['normalize', StandardScaler()],
    ['model', Ridge(alpha=ridge_cv['model'].alpha_)],
])


def metrics(model, train_X, train_y, holdout_X, holdout_y, *, subset=None):
    if subset is not None:
        train_X = train_X[subset]
        holdout_X = holdout_X[subset]
        n_features = len(train_X.columns)
        model.fit(train_X, train_y)
    else:
        model.fit(train_X, train_y)
        if isinstance(model, Pipeline):
            n_features = sum(model['model'].coef_ != 0)
        else:
            n_features = sum(model.coef_ != 0)
    scoring = {'neg_RMSE': 'neg_root_mean_squared_error',
               'neg_MAE': 'neg_mean_absolute_error'}
    np.random.seed(123)
    scores = cross_validate(model, train_X, train_y, cv=5, scoring=scoring)

    train_pred = model.predict(train_X)
    holdout_pred = model.predict(holdout_X)
    return {
        'CV_RMSE': - scores['test_neg_RMSE'].mean(),
        'CV_MAE': - scores['test_neg_MAE'].mean(),
        'train_RMSE': np.sqrt(np.mean((train_y - train_pred)**2)),
        'train_MAE': np.mean(np.abs(train_y - train_pred)),
        'holdout_RMSE': np.sqrt(np.mean((holdout_y - holdout_pred)**2)),
        'holdout_MAE': np.mean(np.abs(holdout_y - holdout_pred)),
        'n_features': n_features,
    }
pd.DataFrame({
    'Full': metrics(car_lm, train_X, train_y, holdout_X, holdout_y),
    'Exhaustive': metrics(efs_model, train_X, train_y, holdout_X, holdout_y, subset=efs_subset),
    'Forward': metrics(sfs_forward_model, train_X, train_y, holdout_X, holdout_y, subset=forward_subset),
    'Backward': metrics(sfs_backward_model, train_X, train_y, holdout_X, holdout_y, subset=backward_subset),
    'Stepwise': metrics(sfs_stepwise_model, train_X, train_y, holdout_X, holdout_y, subset=stepwise_subset),
    'Ridge': metrics(ridge_opt, train_X, train_y, holdout_X, holdout_y),
    'Lasso': metrics(lasso_opt, train_X, train_y, holdout_X, holdout_y),
}).transpose().round(0)

**Cell 23: Detailed statsmodels summary**  
Finally, a full linear regression model is fit using `statsmodels` to obtain detailed summary output, including p‑values, R², and diagnostic statistics. This provides a more traditional statistical perspective on the model.

**Interpretation:** The summary includes coefficient estimates, standard errors, t‑statistics, and p‑values. It confirms which predictors are statistically significant (e.g., Age_08_04, KM, HP, etc.) and provides overall model fit measures (R² ≈ 0.85, adjusted R² ≈ 0.85).

In [ ]:
# run a linear regression of Price on the remaining 11 predictors in the training set
train_df = train_X.join(train_y)

predictors = train_X.columns
formula = 'Price ~ ' + ' + '.join(predictors)

car_lm = sm.ols(formula=formula, data=train_df).fit()
car_lm.summary()